# 🗣 Qwen3-TTS on Kaggle · 2× T4 · driven from your phone

Runs **Qwen3-TTS** ([QwenLM/Qwen3-TTS](https://github.com/QwenLM/Qwen3-TTS)) on a Kaggle **GPU T4 x2** instance and serves a **mobile-first web app** — open a link on your phone and generate speech from anywhere.

**3 modes, 10 languages, auto-detect:**

| Mode | What it does |
| --- | --- |
| 🎙 **Voices** | 9 premium speakers (zh/en/ja/ko, incl. Beijing & Sichuan dialects) + free-text emotion/style instruct |
| ✨ **Design** | describe *any* voice in natural language — "breathy late-night radio female" |
| 🎤 **Clone** | upload a 3–10 s clip (+ optional transcript) and speak in that voice |

### 📱 Phone quick-start
1. Kaggle → notebook settings → Accelerator **GPU T4 x2**, Internet **On**
2. **⋮ menu → Run All** — first run ≈ 5 min (model download), later runs ≈ 1 min (cached)
3. Scroll to **Step 5** → tap the purple `https://…trycloudflare.com` link → done 🎉

> Code cells are saved with *hidden-source* metadata: Jupyter / VS Code / nbviewer show them as slim bars you tap to expand. Kaggle's editor shows code as usual — just **Run All** and follow the step cards; each cell also opens with a `STEP n` banner line.

### The 6 steps

| Step | What happens | You do |
| --- | --- | --- |
| **1 · Install** | deps + hardware check | nothing |
| **2 · Load** | ⚙️ **all settings** + one model replica per T4 | (optional) flip settings |
| **3 · Engine** | chunk → sort → batch → both GPUs | nothing |
| **4 · Web app** | phone UI + API + progress | nothing |
| **5 · Launch** | server + public URL | tap the link |
| **6 · Bench** | speed proof (optional) | skip if you like |

### Why it's fast on 2× T4
Qwen3-TTS decodes autoregressively, so speed comes from feeding both GPUs well:

- **1 replica per GPU** — both T4s synthesize in parallel (data-parallel over sentence batches)
- **Batched decoding** — up to **8 sentences per forward pass** (~3–5× per-GPU throughput on memory-bound T4s)
- **Length-aware batching** — sentences sorted by estimated audio tokens (CJK-aware), so batches don't idle
- **Balanced dispatch** — biggest batches alternate GPUs; auto `max_new_tokens` caps stop runaways
- **fp16 talker + fp32 codec + SDPA** — Turing tensor cores, no flash-attn compile (needs Ampere+)
- **Pipelined CPU** — stitching + MP3/FLAC/Opus encode overlap GPU work; `hf_transfer` + persistent cache in `/kaggle/working` make the 4.5 GB download a one-time event
- **OOM-safe** — batches that don't fit auto-split and retry; clone references are encoded once and cached

## 🧰 Step 1 — Install + hardware check

Pins `qwen-tts`'s real deps (its own pin list would clobber Kaggle's CUDA torch), grabs ffmpeg/sox/cloudflared, and prints what hardware you got. Expect **2× Tesla T4 (16 GB)**. *No settings — just run.*

In [ ]:
# ──────────────────────────────────────────────
# STEP 1 · INSTALL + HARDWARE CHECK
# no settings here — just run it (≈1 min)
# ──────────────────────────────────────────────

import sys, subprocess, os, shutil

def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

# --- Qwen3-TTS package ------------------------------------------------------
# --no-deps: the package's own dependency list would drag in a fresh
# `torchaudio` and can overwrite Kaggle's CUDA-matched torch build.
# We install its real runtime deps ourselves and leave torch alone.
pip("--no-deps", "-U", "qwen-tts")

# pinned exactly as Qwen3-TTS's pyproject.toml requires, minus torch/gradio
pip("transformers==4.57.3", "accelerate==1.12.0", "einops", "sox", "onnxruntime",
    "soundfile", "librosa")

# frontend server + audio encoders + fast HF downloads
pip("fastapi>=0.115", "uvicorn[standard]>=0.30", "pydub", "python-multipart", "hf_transfer")

# system: sox (imported by qwen_tts), ffmpeg (mp3/opus + reference-clip conversion)
subprocess.run(["apt-get", "-qq", "install", "-y", "sox", "ffmpeg"], check=False)

# --- cloudflared quick tunnel (self-contained binary, no login) -------------
if not shutil.which("cloudflared"):
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared",
    ])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

# --- persistent working dir + HF cache --------------------------------------
# /kaggle/working survives notebook re-runs: the 1.7B model (~4.5 GB) is
# downloaded ONCE, every later session starts from the local cache.
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~/qwen3tts_work")
os.makedirs(WORK_DIR, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(WORK_DIR, "hf")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"     # saturate the link on 4.5 GB pulls
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print("deps ok — HF cache:", os.environ["HF_HOME"])

import multiprocessing as mp
import torch

N_GPU = torch.cuda.device_count()
GPUS  = list(range(max(1, N_GPU)))
for g in GPUS[:N_GPU]:
    p = torch.cuda.get_device_properties(g)
    print(f"  cuda:{g} — {p.name}, {p.total_memory/2**30:.1f} GB, sm{p.major}{p.minor}")
if not N_GPU:
    print("  ! no CUDA visible — everything still runs, but on CPU (slow).")
    print("    Kaggle -> Settings -> Accelerator -> GPU T4 x2")

# T4 = Turing (sm75): fp16 has tensor cores, bf16 does not -> fp16 talker.
# Ampere+ (T4's successors) gets bf16. The audio codec is up-cast to fp32
# either way (next cell) for artefact-free vocoding.
if N_GPU:
    HAS_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
    TALKER_DTYPE = torch.bfloat16 if HAS_BF16 else torch.float16
else:
    HAS_BF16 = False
    TALKER_DTYPE = torch.float32

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_grad_enabled(False)

N_CPU = max(2, min(8, mp.cpu_count()))
print(f"{N_GPU} GPU(s) · talker {str(TALKER_DTYPE).split('.')[-1]} · codec fp32 · {N_CPU} usable CPUs")

## ⚙️ Step 2 — Settings + load models

**The only cell you'll ever edit.** Pick model size, enable Design/Clone modes, tune batch size — then it loads **one replica per T4** (weights download once into `/kaggle/working`, cached forever) and warm-compiles both GPUs so your first render is already full-speed.

In [ ]:
# ──────────────────────────────────────────────
# STEP 2 · SETTINGS + LOAD MODELS
# settings at the top · one replica per T4 · warm-up
# ──────────────────────────────────────────────

# ╔═ THE ONLY CELL YOU'LL EVER EDIT ═══════════════════════════════
# ╚ settings apply when this cell runs; re-run it to change them ═

MODEL_SIZE = "1.7B"        # "1.7B" (best) or "0.6B" (~2.5x faster, lighter)
LOAD_MODES = {             # flip to True to enable a mode (+~4.5 GB download)
    "custom_voice": True,  #   9 preset speakers + style/emotion instruct
    "voice_design": False, #   describe ANY voice in natural language
    "voice_clone" : False, #   clone from a 3-10 s reference clip
}

# throughput knobs (T4 sweet spots — safe to leave as-is)
BATCH_PER_GPU = 8      # sentences per forward batch (6-12)
TARGET_CHARS  = 280    # merge short sentences up to this length
MAX_CHARS     = 450    # hard chunk ceiling (~35 s of audio)

import gc, os, time

# --- self-heal: works even if Step 1 was skipped after a kernel restart -----
try: GPUS
except NameError:
    import torch as _t, multiprocessing as _mp
    N_GPU = _t.cuda.device_count(); GPUS = list(range(max(1, N_GPU)))
    TALKER_DTYPE = (_t.bfloat16 if N_GPU and _t.cuda.get_device_capability(0)[0] >= 8
                    else _t.float16 if N_GPU else _t.float32)
    N_CPU = max(2, min(8, _mp.cpu_count()))
    try: WORK_DIR
    except NameError:
        WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") \
                   else os.path.expanduser("~/qwen3tts_work")
        os.makedirs(WORK_DIR, exist_ok=True)
    os.environ.setdefault("HF_HOME", os.path.join(WORK_DIR, "hf"))
    print(f"(step 1 skipped — detected {N_GPU} GPU(s), falling back to defaults)")

from qwen_tts import Qwen3TTSModel

QWEN_MODELS = {
    ("1.7B", "custom_voice"): "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    ("1.7B", "voice_design"): "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    ("1.7B", "voice_clone") : "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    ("0.6B", "custom_voice"): "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
    ("0.6B", "voice_clone") : "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
}
plan = {m: QWEN_MODELS[(MODEL_SIZE, m)]
        for m in ("custom_voice", "voice_design", "voice_clone")
        if LOAD_MODES.get(m) and (MODEL_SIZE, m) in QWEN_MODELS}
if "custom_voice" not in plan:                       # UI needs at least one mode
    plan["custom_voice"] = QWEN_MODELS[(MODEL_SIZE, "custom_voice")]
per_model_gb = 4.5 if MODEL_SIZE == "1.7B" else 2.0
print(f"loading {len(plan)} model(s) x {len(GPUS)} GPU(s): " +
      ", ".join(f"{m}({r.split('-')[-1]})" for m, r in plan.items()))
print(f"  first run downloads ~{per_model_gb*len(plan):.1f} GB into {os.environ['HF_HOME']} "
      f"(cached for every later session)\n")

def _load_replica(repo_id: str, gpu):
    dev = f"cuda:{gpu}" if N_GPU else "cpu"
    rep, last = None, None
    for attn in ("sdpa", "eager"):     # flash-attn needs Ampere+; SDPA is the T4 path
        try:
            rep = Qwen3TTSModel.from_pretrained(
                repo_id, device_map=dev, dtype=TALKER_DTYPE, attn_implementation=attn)
            break
        except Exception as e:
            last = e
    if rep is None:
        raise last
    # fp32 audio codec -> no fp16 ringing in the waveform decoder (cheap: 0.7B params)
    try:
        rep.model.speech_tokenizer.model.float()
    except Exception:
        pass
    rep.model.eval()
    return rep

REPLICAS = {}
for mode, repo in plan.items():
    per = {}
    for g in GPUS:
        t0 = time.time()
        per[g] = _load_replica(repo, g)
        print(f"  {mode:13s} on {'cuda:'+str(g) if N_GPU else 'cpu':8s} "
              f"ready in {time.time()-t0:5.1f}s (weights shared from the HF cache)")
        gc.collect()
    REPLICAS[mode] = per
    if N_GPU:
        torch.cuda.empty_cache()

# --- warm-up: compile the kernels now, not on your first render -------------
t_all = time.time()
for mode, per in REPLICAS.items():
    if mode == "voice_clone":
        continue                        # needs a reference clip; warms on first use
    for g, rep in per.items():
        t0 = time.time()
        if mode == "custom_voice":
            rep.generate_custom_voice(text=["Warm-up.", "All systems go."],
                                      speaker="Aiden", language="English")
        else:
            rep.generate_voice_design(text=["Warm-up."],
                                      instruct="a calm, friendly female voice",
                                      language="English")
        if N_GPU:
            torch.cuda.synchronize(g)
        print(f"  warm-up {mode:13s} {'cuda:'+str(g) if N_GPU else 'cpu':8s} {time.time()-t0:5.1f}s")
print(f"warm-up total: {time.time()-t_all:.1f}s — first render will run at full speed")

# --- speaker / language catalogue for the UI --------------------------------
_SPEAKER_META = [
    ("Vivian",    "Vivian",   "F", "Chinese",           "", "Bright, slightly edgy young female voice."),
    ("Serena",    "Serena",   "F", "Chinese",           "", "Warm, gentle young female voice."),
    ("Uncle_Fu",  "Uncle Fu", "M", "Chinese",           "", "Seasoned male voice, low mellow timbre."),
    ("Dylan",     "Dylan",    "M", "Chinese (Beijing)", "Beijing dialect", "Youthful Beijing male, clear and natural."),
    ("Eric",      "Eric",     "M", "Chinese (Sichuan)", "Sichuan dialect", "Lively Chengdu male, husky brightness."),
    ("Ryan",      "Ryan",     "M", "English",           "", "Dynamic male voice with strong rhythmic drive."),
    ("Aiden",     "Aiden",    "M", "English",           "", "Sunny American male, clear midrange."),
    ("Ono_Anna",  "Ono Anna", "F", "Japanese",          "", "Playful female, light nimble timbre."),
    ("Sohee",     "Sohee",    "F", "Korean",            "", "Warm Korean female, rich emotion."),
]
try:
    _supported = set(REPLICAS["custom_voice"][GPUS[0]].get_supported_speakers() or [])
except Exception:
    _supported = {s[0].lower() for s in _SPEAKER_META}
SPEAKERS = [{"id": sid, "name": name, "gender": gdr, "lang_label": lang, "dialect": dia, "desc": desc}
            for sid, name, gdr, lang, dia, desc in _SPEAKER_META if sid.lower() in _supported]
try:
    _langs = [l for l in REPLICAS["custom_voice"][GPUS[0]].get_supported_languages() or []]
except Exception:
    _langs = ["auto"]
_ORDER = ["Auto", "Chinese", "English", "Japanese", "Korean", "German", "French",
          "Russian", "Portuguese", "Spanish", "Italian"]
_CAP = {l.lower(): l for l in _ORDER}
LANGUAGES = [(_CAP.get(l, l.capitalize())) for l in _langs]
LANGUAGES = [l for l in _ORDER if l in LANGUAGES] + [l for l in LANGUAGES if l not in _ORDER]

if N_GPU:
    for g in GPUS:
        print(f"  cuda:{g} VRAM in use: {torch.cuda.memory_reserved(g)/2**30:.1f} GB")
print(f"ready — {len(SPEAKERS)} speakers, languages: {', '.join(LANGUAGES)}")

## ⚡ Step 3 — Synthesis engine

Script → sentence chunks → length-sorted batches → **both GPUs** → codec decode → CPU stitch/encode. Handles cancellation, OOM retries and clone-prompt caching. *Just run.*

In [ ]:
# ──────────────────────────────────────────────
# STEP 3 · SYNTHESIS ENGINE
# just run — chunk → sort → batch → both GPUs
# ──────────────────────────────────────────────

# ---------------------------------------------------------------------------
# The parallel synthesis engine.
#
#   chunking -> length-sorted batches -> one Qwen3-TTS replica per T4
#   -> batched AR generation (KV-cached) -> GPU codec decode -> CPU stitch/encode
#
# Why this is fast on 2x T4:
#   * every sentence batch runs through ONE generate() call -- batch-8 decoding
#     amortises weight-loading across the batch, ~3-5x the serial throughput
#     per GPU (T4s are memory-bandwidth bound; batching hides that)
#   * batches are composed of similar-length sentences (sorted by estimated
#     audio tokens), so the per-batch wall time = longest member, no waste
#   * two worker threads keep BOTH GPUs busy at all times (big batches first,
#     then alternating, so the GPUs finish together)
#   * mp3/opus/flac encoding runs on a CPU thread-pool while GPUs keep working
# ---------------------------------------------------------------------------
import re, io, threading, time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import numpy as np
import torch

SR = 24000            # Qwen3-TTS-Tokenizer-12Hz codec output rate (fixed)

# --------------------------------------------------------------------------
# 1. Script -> sentence chunks (CJK aware)
# --------------------------------------------------------------------------
# Any newline is a hard boundary; sentence enders (western + CJK) split inside
# paragraphs; short fragments are glued back together; over-long pieces are
# split at commas / colons / spaces.
_SENT_SPLIT = re.compile(r"(?<=[.!?…。！？])\s+")     # western: ender + whitespace
_CJK_END     = re.compile(r"(?<=[。！？；])")            # CJK enders split without spaces

def _sentences(para: str) -> list[str]:
    para = _SENT_SPLIT.sub("\x00", para)
    para = _CJK_END.sub("\x00", para)
    return [s.strip() for s in para.split("\x00") if s.strip()]
_CJK = re.compile(r"[\u2e80-\u9fff\uac00-\ud7af\u3040-\u30ff\u31f0-\u31ff]")

def split_script(text: str, target: int = None, hard_max: int = None) -> list[str]:
    target  = target  or TARGET_CHARS
    hard_max = hard_max or MAX_CHARS
    text = (text or "").replace("\r\n", "\n").strip()
    if not text:
        return []
    paras: list[str] = []
    for para in re.split(r"\n\s*\n|\n", text):          # any newline = break
        para = para.strip()
        if not para:
            continue
        # sentence-split inside the paragraph
        pieces = _sentences(para)
        merged: list[str] = []
        for p in pieces:
            if merged and (len(merged[-1]) + len(p) + 1) <= min(target, 140):
                merged[-1] = f"{merged[-1]} {p}"          # tiny fragments -> glue
            else:
                merged.append(p)
        paras.extend(merged)

    # split anything still over hard_max at commas / colons / spaces
    out: list[str] = []
    for p in paras:
        while len(p) > hard_max:
            cut = -1
            for m in re.finditer(r"[,;:、，；：]|\s", p[:hard_max]):
                cut = m.end()
            cut = cut if cut > hard_max // 3 else hard_max
            out.append(p[:cut].strip())
            p = p[cut:].strip()
        if p:
            out.append(p)
    return out

def _cjk_ratio(s: str) -> float:
    if not s:
        return 0.0
    return len(_CJK.findall(s)) / len(s)

def est_audio_tokens(s: str) -> int:
    """Codec runs at 12.5 tokens/s; English ~15 chars/s, CJK ~4.5 chars/s."""
    cjk = _cjk_ratio(s)
    per_char = 0.85 * (1 - cjk) + 2.8 * cjk
    return int(len(s) * per_char) + 24

def est_audio_seconds(s: str) -> float:
    return est_audio_tokens(s) / 12.5

# --------------------------------------------------------------------------
# 2. Chunks -> length-similar batches
# --------------------------------------------------------------------------
def plan_batches(chunks: list[str], batch_size: int) -> list[list[int]]:
    order = sorted(range(len(chunks)), key=lambda i: -est_audio_tokens(chunks[i]))
    return [order[i:i + batch_size] for i in range(0, len(order), batch_size)]

def _batch_max_new_tokens(chunks: list[str], idxs: list[int]) -> int:
    est = max(est_audio_tokens(chunks[i]) for i in idxs)
    return int(min(6000, max(256, est * 1.5 + 96)))       # codec caps at 8000 positions

# --------------------------------------------------------------------------
# 3. Worker per GPU.  OOM-safe: a batch that doesn't fit is split + retried.
# --------------------------------------------------------------------------
class Cancelled(Exception):
    pass

@dataclass
class _Ctx:
    mode: str
    speaker: str = ""
    instruct: str = ""
    language: str = "Auto"
    voice_clone_prompt: object = None

def _run_batch(rep, texts: list[str], ctx: _Ctx, mnt: int) -> list[np.ndarray]:
    try:
        return _run_batch_inner(rep, texts, ctx, mnt)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(texts) > 1:
            mid = len(texts) // 2
            return (_run_batch(rep, texts[:mid], ctx, mnt)
                    + _run_batch(rep, texts[mid:], ctx, mnt))
        raise

def _run_batch_inner(rep, texts: list[str], ctx: _Ctx, mnt: int) -> list[np.ndarray]:
    if ctx.mode == "custom_voice":
        wavs, sr = rep.generate_custom_voice(
            text=texts, speaker=ctx.speaker or "Aiden", language=ctx.language,
            instruct=ctx.instruct or None, max_new_tokens=mnt,
        )
    elif ctx.mode == "voice_design":
        wavs, sr = rep.generate_voice_design(
            text=texts, instruct=ctx.instruct or "a warm, friendly voice",
            language=ctx.language, max_new_tokens=mnt,
        )
    elif ctx.mode == "voice_clone":
        wavs, sr = rep.generate_voice_clone(
            text=texts, language=ctx.language,
            voice_clone_prompt=ctx.voice_clone_prompt,
            non_streaming_mode=True, max_new_tokens=mnt,
        )
    else:
        raise ValueError(f"unknown mode {ctx.mode}")
    return [np.asarray(w, dtype=np.float32) for w in wavs]

# --------------------------------------------------------------------------
# 4. Voice-clone prompt cache (encode the 3s reference ONCE, reuse forever)
# --------------------------------------------------------------------------
_CLONE_PROMPTS: dict[tuple, list] = {}
_CLONE_LOCK = threading.Lock()

def get_clone_prompt(ref_id: str, ref_path: str, ref_text: str, xvec_only: bool):
    key = (ref_id, bool(xvec_only))
    with _CLONE_LOCK:
        if key not in _CLONE_PROMPTS:
            rep = REPLICAS["voice_clone"][GPUS[0]]
            items = rep.create_voice_clone_prompt(
                ref_audio=ref_path, ref_text=(None if xvec_only else (ref_text or "")),
                x_vector_only_mode=bool(xvec_only))
            _CLONE_PROMPTS[key] = items
            if len(_CLONE_PROMPTS) > 8:                   # keep it small
                _CLONE_PROMPTS.pop(next(iter(_CLONE_PROMPTS)))
        return _CLONE_PROMPTS[key]

# --------------------------------------------------------------------------
# 5. The pipeline
# --------------------------------------------------------------------------
_eng_lock = threading.Lock()          # one job at a time keeps both GPUs fed
_old_pool = globals().get("_enc_pool")          # notebook re-run hygiene
if _old_pool is not None:
    try: _old_pool.shutdown(wait=False, cancel_futures=True)
    except Exception: pass
_enc_pool = ThreadPoolExecutor(max_workers=max(2, N_CPU // 2), thread_name_prefix="enc")

def synth(text: str, *, mode: str = "custom_voice", speaker: str = "Aiden",
          instruct: str = "", language: str = "Auto", gap_ms: int = 160,
          ref: dict | None = None, on_progress=None, cancel=None) -> tuple[int, np.ndarray, int]:
    chunks = split_script(text)
    if not chunks:
        raise ValueError("text is empty")
    if mode not in REPLICAS:
        raise ValueError(f"model for '{mode}' is not loaded (see the model-loading cell)")
    total = len(chunks)
    batches = plan_batches(chunks, BATCH_PER_GPU)

    ctx = _Ctx(mode=mode, speaker=speaker, instruct=instruct, language=language or "Auto")
    if mode == "voice_clone":
        if not ref or not ref.get("path"):
            raise ValueError("voice clone needs a reference clip (upload one in the Clone tab)")
        ctx.voice_clone_prompt = get_clone_prompt(
            ref["id"], ref["path"], ref.get("text") or "", ref.get("xvec_only", False))

    results: list[np.ndarray | None] = [None] * total
    progress = {"done": 0}
    plock = threading.Lock()
    errors: list[Exception] = []

    def worker(gpu: int, my_batches: list[list[int]]):
        try:
            rep = REPLICAS[mode][gpu]
            for b in my_batches:
                if cancel is not None and cancel():
                    raise Cancelled()
                wavs = _run_batch(rep, [chunks[i] for i in b], ctx,
                                  _batch_max_new_tokens(chunks, b))
                for i, w in zip(b, wavs):
                    results[i] = w
                with plock:
                    progress["done"] += len(b)
                    if on_progress:
                        on_progress(progress["done"], total)
        except Cancelled:
            pass
        except Exception as e:                            # surface, keep sibling GPU alive
            errors.append(e)

    # big batches first, then alternate GPUs -> both T4s finish together
    assignment: dict[int, list[list[int]]] = {g: [] for g in GPUS}
    for i, b in enumerate(batches):
        assignment[GPUS[i % len(GPUS)]].append(b)

    with _eng_lock:
        threads = [threading.Thread(target=worker, args=(g, bs), daemon=True)
                   for g, bs in assignment.items() if bs]
        for t in threads: t.start()
        for t in threads: t.join()

    for e in errors:
        raise e
    if cancel is not None and cancel():
        raise Cancelled()
    if any(r is None for r in results):
        raise RuntimeError("internal: some chunks produced no audio")

    gap = np.zeros(int(SR * max(0, gap_ms) / 1000), dtype=np.float32)
    stitched = [results[0]]
    for r in results[1:]:
        stitched.append(gap); stitched.append(r)
    wav = np.concatenate(stitched).astype(np.float32)
    return SR, wav, total

# --------------------------------------------------------------------------
# 6. Encode off the request path (WAV / MP3 / FLAC / Opus)
# --------------------------------------------------------------------------
def encode(sr: int, wav: np.ndarray, fmt: str) -> tuple[bytes, str]:
    fmt = (fmt or "wav").lower()
    wav = np.clip(wav, -1.0, 1.0)
    buf = io.BytesIO()
    if fmt == "mp3":
        from pydub import AudioSegment
        pcm = (wav * 32767).astype(np.int16).tobytes()
        AudioSegment(pcm, frame_rate=sr, sample_width=2, channels=1)\
            .export(buf, format="mp3", bitrate="160k")
        return buf.getvalue(), "audio/mpeg"
    import soundfile as sf
    if fmt == "opus":
        sf.write(buf, wav, sr, format="OGG", subtype="OPUS"); return buf.getvalue(), "audio/ogg"
    if fmt == "flac":
        sf.write(buf, wav, sr, format="FLAC"); return buf.getvalue(), "audio/flac"
    sf.write(buf, wav, sr, format="WAV", subtype="PCM_16"); return buf.getvalue(), "audio/wav"

def encode_async(sr: int, wav: np.ndarray, fmt: str):
    return _enc_pool.submit(encode, sr, wav, fmt)

# ---- sanity check ---------------------------------------------------------
if "REPLICAS" not in globals() or "custom_voice" not in globals().get("REPLICAS", {}):
    print("⚠ run Step 2 first (no models loaded) — engine defined, sanity check skipped")
else:
    t0 = time.time()
    _sr, _w, _n = synth("Qwen3 TTS is ready. Both GPUs are warm, batches are balanced.",
                        mode="custom_voice", speaker="Aiden", language="English")
    print(f"engine ok: {_n} chunks -> {len(_w)/_sr:.2f}s audio in {time.time()-t0:.2f}s wall "
          f"(RTF {(len(_w)/_sr)/(time.time()-t0):.2f})")

## 📱 Step 4 — Phone web app (UI + API)

The mobile page (bottom-sheet speaker picker, emotion chips, clone upload, progress bar, player, history, PWA) plus the FastAPI job queue behind it. *Just run.*

In [ ]:
# ──────────────────────────────────────────────
# STEP 4 · PHONE WEB APP — UI + API
# just run — mobile page, job queue, SSE progress, ref uploads
# ──────────────────────────────────────────────

INDEX_HTML = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover, maximum-scale=5">
<meta name="theme-color" content="#0b0d10">
<meta name="apple-mobile-web-app-capable" content="yes">
<meta name="apple-mobile-web-app-status-bar-style" content="black-translucent">
<meta name="apple-mobile-web-app-title" content="Qwen3 TTS">
<meta name="mobile-web-app-capable" content="yes">
<link rel="manifest" href="/manifest.webmanifest">
<link rel="icon" href="/icon.svg" type="image/svg+xml">
<link rel="apple-touch-icon" href="/icon.svg">
<title>Qwen3-TTS · 2×T4</title>
<style>
:root{
  --bg:#0b0d10; --surface:#14181e; --surface-2:#1b2029; --line:#252b36;
  --text:#e8ecf1; --muted:#9aa3b2; --accent:#8b7cf7; --accent-2:#6d5df6;
  --good:#4ade80; --warn:#f59e0b; --bad:#ef4444;
  --r:14px; --r-sm:10px; --tap:48px;
  --sat:env(safe-area-inset-top); --sab:env(safe-area-inset-bottom);
  --sal:env(safe-area-inset-left); --sar:env(safe-area-inset-right);
}
@media(prefers-color-scheme: light){
  :root{ --bg:#f6f7fb; --surface:#ffffff; --surface-2:#eef1f6; --line:#dfe4ec;
         --text:#0e1420; --muted:#5b6473; --accent:#6d5df6; --accent-2:#5a49e8; }
}
*{ box-sizing:border-box; -webkit-tap-highlight-color:transparent; }
html,body{ margin:0; padding:0; background:var(--bg); color:var(--text);
  font: 16px/1.45 -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue",
        Arial, "Noto Sans", sans-serif; -webkit-font-smoothing:antialiased;
  overscroll-behavior-y:none; }
body{ min-height:100dvh; padding-left:var(--sal); padding-right:var(--sar);
  padding-bottom: calc(var(--sab) + 92px); }
button, input, textarea, select{ font:inherit; color:inherit; }
button{ background:none; border:0; cursor:pointer; }

/* header */
header{ position:sticky; top:0; z-index:5; background:color-mix(in oklab, var(--bg) 88%, transparent);
  backdrop-filter:saturate(140%) blur(10px); -webkit-backdrop-filter:saturate(140%) blur(10px);
  padding: calc(var(--sat) + 10px) 16px 10px; border-bottom:1px solid var(--line);
  display:flex; align-items:center; gap:10px; }
.logo{ width:32px; height:32px; border-radius:9px; background:
  linear-gradient(135deg, var(--accent), var(--accent-2)); display:grid; place-items:center;
  color:#fff; font-weight:800; font-size:14px; }
.title{ font-weight:700; font-size:16px; letter-spacing:.2px; }
.sub{ font-size:11px; color:var(--muted); margin-top:1px; }
.pill{ margin-left:auto; display:inline-flex; align-items:center; gap:6px;
  padding:6px 10px; background:var(--surface-2); border:1px solid var(--line);
  border-radius:999px; font-size:12px; color:var(--muted); white-space:nowrap; }
.dot{ width:8px; height:8px; border-radius:50%; background:var(--good); box-shadow:0 0 0 3px color-mix(in oklab, var(--good) 30%, transparent);}

/* main */
main{ max-width: 680px; margin: 0 auto; padding: 12px 14px; display:flex; flex-direction:column; gap:12px; }
.card{ background:var(--surface); border:1px solid var(--line); border-radius:var(--r); padding:14px; }
.label{ font-size:12px; color:var(--muted); text-transform:uppercase; letter-spacing:.6px; margin-bottom:8px;
  display:flex; justify-content:space-between; }

/* mode tabs */
.tabs{ display:grid; grid-auto-flow:column; grid-auto-columns:1fr; gap:6px; background:var(--surface-2);
  padding:4px; border-radius:12px; border:1px solid var(--line); }
.tabs button{ min-height:42px; border-radius:9px; font-weight:600; font-size:14px; color:var(--muted); }
.tabs button.on{ background:var(--surface); color:var(--text); box-shadow:0 1px 2px rgba(0,0,0,.15); }

/* voice card */
.voice{ display:flex; align-items:center; gap:12px; padding:14px; min-height:var(--tap); width:100%; text-align:left; }
.voice .avatar{ width:44px; height:44px; border-radius:50%;
  background:linear-gradient(135deg, var(--accent), var(--accent-2));
  color:#fff; display:grid; place-items:center; font-weight:700; }
.voice .who{ flex:1; min-width:0; }
.voice .who b{ display:block; font-size:16px; }
.voice .who span{ font-size:12px; color:var(--muted); display:block; white-space:nowrap; overflow:hidden; text-overflow:ellipsis;}
.voice .chev{ color:var(--muted); font-size:22px; }

/* script */
textarea{ width:100%; min-height:38vh; max-height:65vh; background:var(--surface);
  border:1px solid var(--line); border-radius:var(--r); padding:14px; color:var(--text);
  font-size:16px; line-height:1.5; resize:vertical; }
textarea:focus{ outline:2px solid var(--accent); outline-offset:1px; }
textarea.small{ min-height:17vh; }
.meter{ display:flex; justify-content:space-between; font-size:12px; color:var(--muted); padding:6px 4px 0; }

/* controls */
.row{ display:grid; grid-template-columns: 1fr auto; gap:10px; align-items:center; margin-bottom:12px; }
.row:last-child{ margin-bottom:0; }
.row .name{ font-size:14px; }
.val{ font-variant-numeric: tabular-nums; color:var(--muted); font-size:13px; min-width:56px; text-align:right; }
input[type=range]{ -webkit-appearance:none; appearance:none; width:100%; height:36px; background:transparent; }
input[type=range]::-webkit-slider-runnable-track{ height:6px; border-radius:3px; background:var(--surface-2); }
input[type=range]::-moz-range-track{ height:6px; border-radius:3px; background:var(--surface-2); }
input[type=range]::-webkit-slider-thumb{ -webkit-appearance:none; appearance:none; width:26px; height:26px; border-radius:50%;
  background:var(--accent); margin-top:-10px; border:3px solid var(--bg); box-shadow:0 2px 6px rgba(0,0,0,.3);}
input[type=range]::-moz-range-thumb{ width:22px; height:22px; border-radius:50%; background:var(--accent); border:3px solid var(--bg); }

select{ width:100%; min-height:46px; background:var(--surface-2); border:1px solid var(--line);
  border-radius:10px; padding:0 12px; color:var(--text); font-size:15px; -webkit-appearance:none; appearance:none;
  background-image:linear-gradient(45deg,transparent 50%,var(--muted) 50%),linear-gradient(135deg,var(--muted) 50%,transparent 50%);
  background-position:calc(100% - 18px) 50%, calc(100% - 13px) 50%; background-size:5px 5px,5px 5px; background-repeat:no-repeat; }

.segmented{ display:grid; grid-auto-flow:column; grid-auto-columns:1fr; gap:6px; background:var(--surface-2); padding:4px;
  border-radius:12px; border:1px solid var(--line); }
.segmented button{ min-height:40px; border-radius:9px; font-weight:600; font-size:13px; color:var(--muted); }
.segmented button.on{ background:var(--surface); color:var(--text); box-shadow:0 1px 2px rgba(0,0,0,.15); }

/* chips */
.chips{ display:flex; gap:8px; overflow-x:auto; padding:2px 0 10px; scrollbar-width:none; -ms-overflow-style:none; }
.chips::-webkit-scrollbar{ display:none; }
.chip{ flex:0 0 auto; min-height:36px; padding:0 14px; border-radius:999px; background:var(--surface-2);
  border:1px solid var(--line); font-size:13px; font-weight:600; color:var(--muted); }
.chip.on{ background:color-mix(in oklab, var(--accent) 22%, var(--surface)); border-color:var(--accent); color:var(--text); }

/* clone upload */
.upl{ display:flex; gap:10px; align-items:center; }
.upl label{ flex:1; min-height:48px; border:1.5px dashed var(--line); border-radius:12px; display:flex;
  align-items:center; justify-content:center; gap:8px; font-size:14px; font-weight:600; color:var(--muted); }
.upl input{ display:none; }
.upl .rec{ width:48px; height:48px; border-radius:12px; background:var(--surface-2); border:1px solid var(--line);
  display:grid; place-items:center; font-size:20px; }
.refmeta{ font-size:12px; color:var(--muted); margin-top:8px; display:flex; align-items:center; gap:8px; flex-wrap:wrap; }
.refmeta b{ color:var(--good); font-weight:600; }
audio.mini{ width:100%; margin-top:8px; }

.toggle{ display:flex; align-items:center; gap:10px; padding:10px 0 2px; font-size:14px; }
.toggle input{ width:22px; height:22px; accent-color:var(--accent); }
.toggle span{ color:var(--muted); font-size:12px; display:block; }

/* sticky action */
.fab-wrap{ position:fixed; left:0; right:0; bottom:0; padding: 10px 14px calc(var(--sab) + 10px);
  background: linear-gradient(to top, var(--bg) 55%, transparent);
  z-index:4; }
.fab-wrap .inner{ max-width:680px; margin:0 auto; display:flex; gap:10px; align-items:center; }
.fab{ flex:1; min-height:56px; background:var(--accent); color:#fff; border-radius:14px;
  font-weight:700; font-size:16px; letter-spacing:.2px; display:inline-flex; align-items:center; justify-content:center; gap:10px;
  box-shadow: 0 6px 20px color-mix(in oklab, var(--accent) 40%, transparent); transition: transform .06s ease; }
.fab:active{ transform: scale(.98); }
.fab[disabled]{ background:var(--surface-2); color:var(--muted); box-shadow:none; }
.fab .spinner{ width:18px; height:18px; border-radius:50%; border:2px solid rgba(255,255,255,.4); border-top-color:#fff; animation:spin .8s linear infinite; }
@keyframes spin{ to{ transform:rotate(360deg); } }
.icon-btn{ width:56px; height:56px; border-radius:14px; background:var(--surface); border:1px solid var(--line); display:grid; place-items:center; color:var(--text); }

/* progress bar */
.progress{ position:fixed; top:0; left:0; right:0; height:3px; background:transparent; z-index:10; pointer-events:none; }
.progress .bar{ height:100%; width:0%; background:linear-gradient(90deg, var(--accent), var(--accent-2)); transition: width .2s ease; }

/* audio result */
.result{ display:none; }
.result.on{ display:block; }
.result audio{ width:100%; margin-top:8px; }
.result .stats{ display:flex; gap:12px; font-size:12px; color:var(--muted); flex-wrap:wrap; margin-top:8px; }
.result .stats b{ color:var(--text); font-weight:600; }
.result .dlrow{ display:flex; gap:8px; margin-top:10px; }
.btn{ flex:1; min-height:44px; background:var(--surface-2); border:1px solid var(--line);
  border-radius:10px; font-weight:600; font-size:14px; color:var(--text); display:inline-flex; align-items:center; justify-content:center; gap:8px; text-decoration:none; }
.btn.primary{ background:var(--accent); color:#fff; border-color:transparent; }

/* history */
.hist-item{ display:flex; align-items:center; gap:10px; padding:10px; border-radius:10px; }
.hist-item + .hist-item{ border-top:1px solid var(--line); border-radius:0; }
.hist-item .who{ flex:1; min-width:0; }
.hist-item .who b{ display:block; font-size:14px; }
.hist-item .who span{ font-size:12px; color:var(--muted); white-space:nowrap; overflow:hidden; text-overflow:ellipsis; display:block; }
.hist-item .play{ width:40px; height:40px; border-radius:50%; background:var(--accent); color:#fff; display:grid; place-items:center; }

/* bottom sheet */
.sheet-back{ position:fixed; inset:0; background:rgba(0,0,0,.55); opacity:0; pointer-events:none; transition:opacity .18s ease; z-index:20; }
.sheet-back.on{ opacity:1; pointer-events:auto; }
.sheet{ position:fixed; left:0; right:0; bottom:0; z-index:21; background:var(--surface); border-top-left-radius:20px; border-top-right-radius:20px;
  transform: translateY(100%); transition: transform .22s cubic-bezier(.2,.8,.2,1);
  max-height: 88dvh; display:flex; flex-direction:column; padding-bottom: var(--sab); }
.sheet.on{ transform: translateY(0); }
.sheet .grabber{ width:44px; height:5px; background:var(--line); border-radius:3px; margin: 8px auto 4px; }
.sheet .head{ display:flex; align-items:center; padding: 6px 14px 10px; gap:10px; border-bottom:1px solid var(--line); }
.sheet .head b{ font-size:16px; }
.sheet .head button{ margin-left:auto; color:var(--muted); font-size:15px; min-height:44px; padding: 0 10px; }
.sheet .search{ padding: 10px 14px; border-bottom:1px solid var(--line); }
.sheet .search input{ width:100%; min-height:44px; padding: 0 12px; background:var(--surface-2); border:1px solid var(--line); border-radius:10px; }
.sheet .list{ overflow-y:auto; padding: 6px 8px 8px; }
.sheet .group{ font-size:11px; color:var(--muted); text-transform:uppercase; letter-spacing:.7px; padding: 12px 8px 6px; }
.sheet .item{ display:flex; align-items:center; gap:12px; padding:12px 10px; border-radius:12px; min-height:var(--tap); width:100%; text-align:left; }
.sheet .item:active{ background: var(--surface-2); }
.sheet .item.on{ background: color-mix(in oklab, var(--accent) 15%, transparent); }
.sheet .item .avatar{ width:36px; height:36px; border-radius:50%; background:linear-gradient(135deg, var(--accent), var(--accent-2)); color:#fff; display:grid; place-items:center; font-weight:700; font-size:13px; }
.sheet .item .meta{ flex:1; min-width:0; }
.sheet .item .meta b{ display:block; font-size:15px; }
.sheet .item .meta span{ font-size:12px; color:var(--muted); display:block; }
.sheet .item .check{ margin-left:auto; color:var(--accent); opacity:0; }
.sheet .item.on .check{ opacity:1; }

/* toast */
.toast{ position:fixed; left:14px; right:14px; bottom: calc(var(--sab) + 110px); z-index:30;
  background:var(--bad); color:#fff; padding:12px 14px; border-radius:12px; box-shadow: 0 8px 30px rgba(0,0,0,.3);
  transform: translateY(20px); opacity:0; transition: all .2s ease; pointer-events:none; text-align:center; }
.toast.on{ transform:none; opacity:1; }

/* utility */
.hide{ display:none !important; }
</style>
</head>
<body>
  <div class="progress"><div class="bar" id="pbar"></div></div>

  <header>
    <div class="logo">Q</div>
    <div>
      <div class="title">Qwen3-TTS</div>
      <div class="sub" id="sub">connecting…</div>
    </div>
    <div class="pill"><span class="dot" id="statusDot"></span><span id="statusText">…</span></div>
  </header>

  <main>
    <div class="tabs" id="modeTabs" role="tablist"></div>

    <!-- VOICE mode -->
    <section id="paneVoice">
      <button class="card voice" id="voiceBtn" aria-label="Choose speaker">
        <div class="avatar" id="vAvatar">A</div>
        <div class="who">
          <b id="vName">Aiden</b>
          <span id="vMeta">English · Male</span>
        </div>
        <div class="chev">›</div>
      </button>

      <div class="card" style="margin-top:12px">
        <div class="label"><span>Style / emotion <i style="text-transform:none;letter-spacing:0">(optional)</i></span></div>
        <div class="chips" id="chips"></div>
        <textarea id="instruct" class="small" style="min-height:11vh" spellcheck="true"
          placeholder="e.g. Speak with an excited, upbeat tone, fast pace."></textarea>
      </div>
    </section>

    <!-- DESIGN mode -->
    <section id="paneDesign" class="hide">
      <div class="card">
        <div class="label"><span>Describe the voice</span><button id="dice" style="color:var(--accent);font-weight:700;font-size:13px">🎲 Surprise me</button></div>
        <textarea id="designDesc" class="small" spellcheck="true"
          placeholder="Describe any voice: gender, age, accent, timbre, pace, emotion…"></textarea>
      </div>
    </section>

    <!-- CLONE mode -->
    <section id="paneClone" class="hide">
      <div class="card">
        <div class="label"><span>Reference clip (3–10 s, clean speech)</span></div>
        <div class="upl">
          <label for="refFile" id="refLabel">🎙️ Record / choose audio</label>
          <input type="file" id="refFile" accept="audio/*">
        </div>
        <div class="refmeta hide" id="refMeta"></div>
        <audio class="mini hide" id="refAudio" controls preload="metadata" playsinline></audio>
        <div class="label" style="margin-top:12px"><span>What the clip says (transcript)</span></div>
        <textarea id="refText" class="small" style="min-height:9vh" spellcheck="true"
          placeholder="Type exactly what is said in the clip — better cloning."></textarea>
        <label class="toggle"><input type="checkbox" id="xvecOnly">
          <span>Skip transcript (timbre-only clone)<span>lower fidelity, but no transcript needed</span></span>
        </label>
      </div>
    </section>

    <!-- shared -->
    <div class="card">
      <div class="label">Language</div>
      <select id="lang"></select>
    </div>

    <div>
      <textarea id="script" spellcheck="true" autocapitalize="sentences"
        enterkeyhint="enter" inputmode="text"
        placeholder="Paste or type your script here…">Qwen3-TTS on two Kaggle T4s. This notebook loads one replica of the model on each GPU, batches your sentences, and keeps both cards busy at once — so long scripts render in seconds, not minutes.</textarea>
      <div class="meter"><span id="charCount">0 chars</span><span id="estAudio">≈ 0s audio</span></div>
    </div>

    <div class="card">
      <div class="row">
        <div class="name">Gap between chunks</div>
        <div class="val" id="gapVal">160 ms</div>
      </div>
      <input type="range" id="gap" min="0" max="600" step="20" value="160">
    </div>

    <div class="card">
      <div class="label">Format</div>
      <div class="segmented" id="fmt" role="radiogroup">
        <button data-v="wav"  class="on">WAV</button>
        <button data-v="mp3">MP3</button>
        <button data-v="flac">FLAC</button>
        <button data-v="opus">Opus</button>
      </div>
    </div>

    <div class="card result" id="result">
      <div class="label"><span>Latest render</span><span id="resStamp"></span></div>
      <audio id="player" controls preload="metadata" playsinline></audio>
      <div class="stats">
        <span><b id="resDur">0.00s</b> audio</span>
        <span>rendered in <b id="resTime">0.00s</b></span>
        <span>RTF <b id="resRTF">0.00×</b></span>
        <span id="resChunks"></span>
      </div>
      <div class="dlrow">
        <a class="btn primary" id="downloadBtn" download="qwen3tts.wav">↓ Download</a>
        <button class="btn" id="shareBtn">Share</button>
      </div>
    </div>

    <div class="card hide" id="histCard">
      <div class="label">History</div>
      <div id="histList"></div>
    </div>
  </main>

  <div class="fab-wrap">
    <div class="inner">
      <button class="icon-btn" id="stopBtn" title="Stop" aria-label="Stop" style="display:none;">■</button>
      <button class="fab" id="goBtn"><span id="goLabel">Generate</span></button>
    </div>
  </div>

  <div class="sheet-back" id="sheetBack"></div>
  <div class="sheet" id="sheet" role="dialog" aria-label="Choose a speaker">
    <div class="grabber"></div>
    <div class="head"><b>Choose a speaker</b><button id="sheetClose">Done</button></div>
    <div class="search"><input id="sheetSearch" placeholder="Search speakers…" enterkeyhint="search"></div>
    <div class="list" id="sheetList"></div>
  </div>

  <div class="toast" id="toast">…</div>

<script>
const $ = s => document.querySelector(s);
const state = { modes: [], mode: 'custom_voice', voices: [], voice: 'Aiden',
                fmt: 'wav', langs: [], lang: 'Auto', busy: false, job: null, ctrl: null,
                history: [], ref: null, chip: '' };

const CHIPS = [
  ['😊 Happy',      'Speak in a very happy, upbeat tone.'],
  ['🔥 Excited',    'Speak with high energy and excitement, fast pace.'],
  ['😌 Calm',       'Speak calmly and warmly, at a relaxed pace.'],
  ['😠 Angry',      'Speak with an angry, irritated tone.'],
  ['😢 Sad',        'Speak sadly and slowly, with a heavy heart.'],
  ['🤫 Whisper',    'Speak in a soft whisper.'],
  ['🎬 Narrator',   'Speak like a deep, cinematic movie-trailer narrator.'],
  ['🃏 Playful',    'Speak with a playful, teasing tone.'],
];
const DESIGN_PRESETS = [
  'A warm middle-aged British male voice, gravelly but kind, like a documentary narrator.',
  'An energetic young female voice with a bright, bubbly tone and quick pace.',
  'A deep, calm male voice with a slow, deliberate documentary pace.',
  'A soft-spoken elderly woman with a gentle, storytelling cadence.',
  'A confident female news anchor with crisp articulation and a neutral accent.',
  'A cheeky teenage boy, informal speech with rising intonation.',
  'A breathy, intimate female voice, late-night radio style.',
  'A booming sports commentator, loud and thrilling.',
];

function toast(msg, kind='bad'){
  const t = $('#toast'); t.textContent = msg;
  t.style.background = kind==='good' ? 'var(--good)' : 'var(--bad)';
  t.classList.add('on'); clearTimeout(toast._t);
  toast._t = setTimeout(()=>t.classList.remove('on'), 3200);
}
function initials(v){ const n=(v.name||v.id).replace(/[^a-z]/gi,''); return (n[0]||'?').toUpperCase(); }

/* ---------- speaker sheet ---------- */
function setVoice(id){
  const v = state.voices.find(x=>x.id===id) || state.voices[0]; if(!v) return;
  state.voice = v.id;
  $('#vName').textContent = v.name;
  $('#vMeta').textContent = `${v.lang_label} · ${v.gender==='F'?'Female':'Male'} · ${v.desc}`;
  $('#vAvatar').textContent = initials(v);
  try{ localStorage.setItem('q3.voice', v.id); }catch{}
  renderSheet($('#sheetSearch').value||'');
}
function renderSheet(q){
  q = (q||'').trim().toLowerCase();
  const list = $('#sheetList'); list.innerHTML='';
  const groups = {};
  for(const v of state.voices){
    if(q && !(v.id.toLowerCase().includes(q) || v.name.toLowerCase().includes(q)
       || v.lang_label.toLowerCase().includes(q) || (v.desc||'').toLowerCase().includes(q))) continue;
    (groups[v.lang_label] ||= []).push(v);
  }
  const order = ['Chinese','English','Japanese','Korean'];
  const names = [...new Set([...order.filter(k=>groups[k]), ...Object.keys(groups)])];
  for(const g of names){
    const h = document.createElement('div'); h.className='group'; h.textContent=g; list.appendChild(h);
    for(const v of groups[g]){
      const b = document.createElement('button'); b.className='item'+(v.id===state.voice?' on':'');
      b.innerHTML = `<span class="avatar">${initials(v)}</span>
        <span class="meta"><b>${v.name}${v.dialect?' · '+v.dialect:''}</b><span>${v.gender==='F'?'Female':'Male'} — ${v.desc}</span></span>
        <span class="check">✓</span>`;
      b.addEventListener('click', ()=>{ setVoice(v.id); closeSheet(); });
      list.appendChild(b);
    }
  }
  if(!names.length){ list.innerHTML = '<div style="padding:20px;text-align:center;color:var(--muted)">No matches.</div>'; }
}
function openSheet(){ $('#sheet').classList.add('on'); $('#sheetBack').classList.add('on');
  setTimeout(()=>$('#sheetSearch').focus({preventScroll:true}), 200); }
function closeSheet(){ $('#sheet').classList.remove('on'); $('#sheetBack').classList.remove('on'); }

/* ---------- mode tabs ---------- */
function setMode(m){
  if(!state.modes.includes(m)) m = state.modes[0];
  state.mode = m;
  for(const b of $('#modeTabs').children) b.classList.toggle('on', b.dataset.m===m);
  $('#paneVoice').classList.toggle('hide', m!=='custom_voice');
  $('#paneDesign').classList.toggle('hide', m!=='voice_design');
  $('#paneClone').classList.toggle('hide', m!=='voice_clone');
  try{ localStorage.setItem('q3.mode', m); }catch{}
  updateMeters();
}

/* ---------- chips ---------- */
function renderChips(){
  const c = $('#chips'); c.innerHTML='';
  for(const [label, text] of CHIPS){
    const b = document.createElement('button'); b.className='chip'+(state.chip===text?' on':'');
    b.textContent = label;
    b.addEventListener('click', ()=>{
      state.chip = (state.chip===text ? '' : text);
      $('#instruct').value = state.chip;
      renderChips();
    });
    c.appendChild(b);
  }
}

/* ---------- format / meters ---------- */
function fmt(v){ state.fmt = v; for(const b of $('#fmt').children) b.classList.toggle('on', b.dataset.v===v);
  $('#downloadBtn').setAttribute('download', 'qwen3tts.'+v);
  try{ localStorage.setItem('q3.fmt', v); }catch{} }

function estSeconds(t){
  let cjk=0; for(const ch of t){ const cp=ch.codePointAt(0);
    if((cp>=0x2e80&&cp<=0x9fff)||(cp>=0xac00&&cp<=0xd7af)||(cp>=0x3040&&cp<=0x30ff)) cjk++; }
  const lat = t.length-cjk;
  return lat/15 + cjk/4.5;
}
function updateMeters(){
  const t = $('#script').value; const c = [...t].length;
  $('#charCount').textContent = `${c.toLocaleString()} chars`;
  const s = estSeconds(t);
  $('#estAudio').textContent = `≈ ${s>=60 ? (s/60).toFixed(1)+'m' : s.toFixed(1)+'s'} audio`;
}

/* ---------- history ---------- */
function pushHistory(item){
  state.history.unshift(item); state.history = state.history.slice(0, 6);
  const list = $('#histList'); list.innerHTML='';
  for(const h of state.history){
    const el = document.createElement('div'); el.className='hist-item';
    el.innerHTML = `<button class="play" aria-label="Play">▶</button>
      <div class="who"><b>${h.label} · ${h.fmt.toUpperCase()}</b><span>${h.preview}</span></div>
      <a class="btn" style="flex:0 0 auto; min-width:64px" href="${h.url}" download="qwen3tts.${h.fmt}">↓</a>`;
    el.querySelector('.play').addEventListener('click', ()=>{ const p=$('#player'); p.src=h.url; p.play(); });
    list.appendChild(el);
  }
  $('#histCard').classList.toggle('hide', !state.history.length);
}

/* ---------- busy ---------- */
function setBusy(b){ state.busy = b;
  $('#goBtn').disabled = b;
  $('#goLabel').innerHTML = b ? '<span class="spinner"></span> Rendering…' : 'Generate';
  $('#stopBtn').style.display = b ? 'grid' : 'none';
  $('#pbar').style.width = b ? '5%' : '0%';
}

/* ---------- status boot ---------- */
async function loadStatus(){
  try{
    const r = await fetch('/api/status'); const j = await r.json();
    state.voices = j.voices || []; state.modes = j.modes || []; state.langs = j.languages || [];
    const tabs = $('#modeTabs'); tabs.innerHTML='';
    const LABELS = {custom_voice:'🎙 Voices', voice_design:'✨ Design', voice_clone:'🎤 Clone'};
    for(const m of state.modes){
      const b = document.createElement('button'); b.dataset.m = m; b.textContent = LABELS[m]||m;
      b.addEventListener('click', ()=>setMode(m));
      tabs.appendChild(b);
    }
    const ls = $('#lang'); ls.innerHTML='';
    for(const l of state.langs){
      const o = document.createElement('option'); o.value=l; o.textContent=(l==='Auto'?'🌐 Auto-detect':l); ls.appendChild(o);
    }
    $('#statusText').textContent = `${j.gpus}× ${j.accelerator}`;
    $('#sub').textContent = `${j.model_size} · ${state.voices.length} speakers · ${j.modes.length} mode${j.modes.length>1?'s':''}`;
    const savedV = localStorage.getItem('q3.voice');
    if(state.voices.length) setVoice(savedV && state.voices.some(v=>v.id===savedV) ? savedV : state.voices[0].id);
    const savedM = localStorage.getItem('q3.mode');
    setMode(savedM && state.modes.includes(savedM) ? savedM : state.modes[0]);
    const savedL = localStorage.getItem('q3.lang'); if(savedL && state.langs.includes(savedL)) ls.value = savedL;
    const savedFmt = localStorage.getItem('q3.fmt'); if(savedFmt) fmt(savedFmt);
    renderSheet(''); renderChips();
  }catch(e){ toast('Backend not reachable'); }
}

/* ---------- reference upload (clone) ---------- */
async function uploadRef(file){
  if(!file) return;
  setBusy(true); $('#refLabel').textContent = '⏫ Uploading…';
  try{
    const fd = new FormData(); fd.append('file', file);
    const r = await fetch('/api/ref', {method:'POST', body:fd});
    if(!r.ok) throw new Error((await r.json().catch(()=>({}))).detail || 'upload failed');
    const j = await r.json();
    state.ref = j;
    $('#refLabel').textContent = '↻ Replace audio';
    const m = $('#refMeta'); m.classList.remove('hide');
    m.innerHTML = `✓ <b>${j.name}</b> · ${j.duration_s.toFixed(1)}s${j.duration_s<2||j.duration_s>15?' · <span style="color:var(--warn)">aim for 3–10s</span>':''}`;
    $('#refAudio').src = `/api/ref/${j.ref_id}/audio`; $('#refAudio').classList.remove('hide');
    toast('Reference ready', 'good');
  }catch(e){ toast(e.message||'Upload failed'); }
  finally{ setBusy(false); }
}

/* ---------- generate ---------- */
function payload(){
  const text = $('#script').value.trim();
  const base = { text, language: state.lang || $('#lang').value || 'Auto',
                 gap_ms: parseInt($('#gap').value,10)||0, format: state.fmt, mode: state.mode };
  if(state.mode==='custom_voice'){
    base.speaker = state.voice;
    base.instruct = $('#instruct').value.trim();
  } else if(state.mode==='voice_design'){
    base.instruct = $('#designDesc').value.trim();
  } else if(state.mode==='voice_clone'){
    if(!state.ref){ toast('Upload a reference clip first'); return null; }
    base.ref_id = state.ref.ref_id;
    base.ref_text = $('#refText').value.trim();
    base.xvec_only = $('#xvecOnly').checked;
  }
  if(!text){ toast('Script is empty'); return null; }
  return base;
}

async function generate(){
  if(state.busy) return;
  const p = payload(); if(!p) return;
  setBusy(true);
  const ctrl = new AbortController(); state.ctrl = ctrl;
  const t0 = performance.now();
  try{
    const jr = await fetch('/api/jobs', {method:'POST', headers:{'content-type':'application/json'},
                                          body: JSON.stringify(p), signal: ctrl.signal});
    if(!jr.ok){ const e = await jr.json().catch(()=>({})); throw new Error(e.detail||'server rejected'); }
    const {job_id} = await jr.json(); state.job = job_id;

    await new Promise((resolve, reject)=>{
      const es = new EventSource('/api/jobs/'+job_id+'/events');
      ctrl.signal.addEventListener('abort', ()=>{ es.close(); reject(new Error('aborted')); });
      es.addEventListener('progress', e=>{
        const d = JSON.parse(e.data);
        $('#pbar').style.width = (5 + 85*(d.done/Math.max(1,d.total))) + '%';
        $('#goLabel').textContent = `Rendering ${d.done}/${d.total}`;
      });
      es.addEventListener('done', e=>{ es.close(); resolve(JSON.parse(e.data)); });
      es.addEventListener('error', e=>{ es.close(); reject(new Error('stream error')); });
      es.addEventListener('err', e=>{ es.close(); reject(new Error(JSON.parse(e.data).message||'failed')); });
    });

    $('#pbar').style.width = '95%';
    const ar = await fetch('/api/jobs/'+job_id+'/audio', {signal: ctrl.signal});
    if(!ar.ok) throw new Error('audio fetch failed');
    const blob = await ar.blob();
    const url  = URL.createObjectURL(blob);
    const meta = JSON.parse(ar.headers.get('X-Meta') || '{}');

    const dur = meta.audio_s || 0, took = (performance.now()-t0)/1000;
    $('#player').src = url;
    $('#resDur').textContent  = dur.toFixed(2)+'s';
    $('#resTime').textContent = took.toFixed(2)+'s';
    $('#resRTF').textContent  = (dur>0 ? (took/dur).toFixed(2) : '-')+'×';
    $('#resChunks').textContent = meta.chunks ? `${meta.chunks} chunk${meta.chunks>1?'s':''} · ${meta.gpus||''} GPU${(meta.gpus||1)>1?'s':''}` : '';
    $('#resStamp').textContent = new Date().toLocaleTimeString();
    const a = $('#downloadBtn'); a.href = url; a.setAttribute('download','qwen3tts.'+state.fmt);
    $('#result').classList.add('on');
    let label = p.mode==='custom_voice' ? (p.speaker||'voice')
              : p.mode==='voice_design' ? 'designed' : 'clone';
    pushHistory({ url, fmt: state.fmt, label, preview: p.text.slice(0,80) });
    $('#pbar').style.width = '100%';
    try{ await $('#player').play(); }catch{}
  }catch(e){
    if(e.name!=='AbortError') toast(e.message||'Generation failed');
  }finally{
    setTimeout(()=>{ if(!state.busy) $('#pbar').style.width='0%'; }, 500);
    state.job=null; state.ctrl=null; setBusy(false);
  }
}
async function stopJob(){
  if(state.ctrl) state.ctrl.abort();
  if(state.job){ try{ await fetch('/api/jobs/'+state.job, {method:'DELETE'}); }catch{} }
}
async function share(){
  const p = $('#player'); if(!p.src) return;
  try{
    const blob = await (await fetch(p.src)).blob();
    const file = new File([blob], 'qwen3tts.'+state.fmt, {type: blob.type});
    if(navigator.canShare && navigator.canShare({files:[file]})){
      await navigator.share({ files:[file], title:'Qwen3-TTS render' });
    }else{
      $('#downloadBtn').click();
    }
  }catch(e){ toast('Share cancelled'); }
}

/* ---------- wire up ---------- */
$('#voiceBtn').addEventListener('click', openSheet);
$('#sheetBack').addEventListener('click', closeSheet);
$('#sheetClose').addEventListener('click', closeSheet);
$('#sheetSearch').addEventListener('input', e=>renderSheet(e.target.value));
$('#fmt').addEventListener('click', e=>{ if(e.target.dataset.v) fmt(e.target.dataset.v); });
$('#gap').addEventListener('input', e=>{ $('#gapVal').textContent = e.target.value+' ms'; });
$('#lang').addEventListener('change', e=>{ state.lang = e.target.value; try{localStorage.setItem('q3.lang', e.target.value);}catch{} });
$('#script').addEventListener('input', updateMeters);
$('#goBtn').addEventListener('click', generate);
$('#stopBtn').addEventListener('click', stopJob);
$('#shareBtn').addEventListener('click', share);
$('#refFile').addEventListener('change', e=>uploadRef(e.target.files[0]));
$('#dice').addEventListener('click', ()=>{
  $('#designDesc').value = DESIGN_PRESETS[Math.floor(Math.random()*DESIGN_PRESETS.length)];
});
$('#instruct').addEventListener('input', ()=>{ if($('#instruct').value !== state.chip){ state.chip=''; renderChips(); } });
// swipe-down on sheet grabber
(()=>{ const s=$('#sheet'); let sy=0, dy=0, drag=false;
  s.addEventListener('touchstart', e=>{ if(e.target.classList.contains('grabber')||e.target.closest('.head')){ drag=true; sy=e.touches[0].clientY; }},{passive:true});
  s.addEventListener('touchmove',  e=>{ if(!drag) return; dy=e.touches[0].clientY-sy; if(dy>0){ s.style.transform=`translateY(${dy}px)`; }},{passive:true});
  s.addEventListener('touchend',   ()=>{ if(!drag) return; drag=false; s.style.transform=''; if(dy>90) closeSheet(); dy=0; });
})();

updateMeters(); loadStatus();
if('serviceWorker' in navigator){ navigator.serviceWorker.register('/sw.js').catch(()=>{}); }
</script>
</body></html>"""

MANIFEST_JSON = '''{
  "name": "Qwen3-TTS · 2x T4",
  "short_name": "Qwen3-TTS",
  "start_url": "/",
  "display": "standalone",
  "background_color": "#0b0d10",
  "theme_color": "#0b0d10",
  "orientation": "portrait",
  "icons": [
    { "src": "/icon.svg", "sizes": "any", "type": "image/svg+xml", "purpose": "any maskable" }
  ]
}'''

ICON_SVG = '''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 512 512">
  <defs><linearGradient id="g" x1="0" y1="0" x2="1" y2="1">
    <stop offset="0" stop-color="#8b7cf7"/><stop offset="1" stop-color="#6d5df6"/></linearGradient></defs>
  <rect width="512" height="512" rx="112" fill="url(#g)"/>
  <text x="50%" y="58%" text-anchor="middle" font-family="-apple-system,Segoe UI,Roboto,sans-serif"
        font-weight="800" font-size="280" fill="#fff">Q</text>
</svg>'''

SW_JS = """
const CACHE = "qwen3tts-v1";
const SHELL = ["/", "/manifest.webmanifest", "/icon.svg"];
self.addEventListener("install", e => {
  e.waitUntil(caches.open(CACHE).then(c => c.addAll(SHELL)).then(() => self.skipWaiting()));
});
self.addEventListener("activate", e => e.waitUntil(
  caches.keys().then(ks => Promise.all(ks.filter(k => k !== CACHE).map(k => caches.delete(k))))
    .then(() => self.clients.claim())
));
self.addEventListener("fetch", e => {
  const u = new URL(e.request.url);
  if (u.pathname.startsWith("/api/")) return;
  e.respondWith(
    fetch(e.request).then(r => {
      const copy = r.clone();
      if (r.ok && e.request.method === "GET" && SHELL.includes(u.pathname))
        caches.open(CACHE).then(c => c.put(e.request, copy));
      return r;
    }).catch(() => caches.match(e.request).then(r => r || caches.match("/")))
  );
});
"""

print(f"frontend assets built ({len(INDEX_HTML):,} bytes html)")

import asyncio, json, uuid, threading, queue as pyqueue, time, os, subprocess
import torch
from dataclasses import dataclass, field
from fastapi import FastAPI, HTTPException, Response, Request, UploadFile, File
from fastapi.responses import HTMLResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware

try: WORK_DIR
except NameError:
    WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~/qwen3tts_work")
    os.makedirs(WORK_DIR, exist_ok=True)

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"], expose_headers=["X-Meta"])

# --------------------------------------------------------------------------
# reference-clip store (voice clone)
# --------------------------------------------------------------------------
REF_DIR = os.path.join(WORK_DIR, "refs")
os.makedirs(REF_DIR, exist_ok=True)
REFS: dict[str, dict] = {}
REFS_LOCK = threading.Lock()

def _save_ref(data: bytes, name: str = "clip") -> dict:
    rid = uuid.uuid4().hex[:12]
    raw = os.path.join(REF_DIR, f"{rid}_in")
    with open(raw, "wb") as f:
        f.write(data)
    wav = os.path.join(REF_DIR, f"{rid}.wav")
    # any phone format (m4a/webm/ogg/…) -> 24 kHz mono wav, what the codec wants
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw, "-ac", "1", "-ar", "24000", wav],
                   check=True, timeout=60)
    os.remove(raw)
    import soundfile as sf
    info = sf.info(wav)
    meta = {"ref_id": rid, "path": wav, "name": os.path.basename(name or "clip"),
            "duration_s": float(info.duration)}
    with REFS_LOCK:
        REFS[rid] = meta
    return meta

def _reap_refs():
    while True:
        time.sleep(600)
        with REFS_LOCK:
            stale = []
            for m in REFS.values():
                try:
                    if time.time() - os.path.getmtime(m["path"]) > 7200:
                        stale.append(m)
                except OSError:
                    stale.append(m)                     # file already gone
            for m in stale:
                REFS.pop(m["ref_id"], None)
                try: os.remove(m["path"])
                except OSError: pass
threading.Thread(target=_reap_refs, daemon=True).start()

# --------------------------------------------------------------------------
# job queue
# --------------------------------------------------------------------------
@dataclass
class Job:
    id: str
    mode: str; text: str; speaker: str; instruct: str; language: str
    gap_ms: int; fmt: str; ref_id: str | None; ref_text: str; xvec_only: bool
    events: pyqueue.Queue = field(default_factory=pyqueue.Queue)
    audio: bytes | None = None
    mime: str = "audio/wav"
    meta: dict = field(default_factory=dict)
    cancelled: bool = False
    error: str | None = None
    started: float = 0.0

JOBS: dict[str, Job] = {}
JOBS_LOCK = threading.Lock()

def _run_job(job: Job):
    try:
        job.started = time.time()
        def prog(done, total):
            job.events.put(("progress", {"done": done, "total": total, "phase": "synthesizing"}))
        ref = None
        if job.mode == "voice_clone":
            with REFS_LOCK:
                r = REFS.get(job.ref_id)
            if r is None:
                raise ValueError("reference clip expired — upload it again")
            ref = {"id": r["ref_id"], "path": r["path"], "text": job.ref_text, "xvec_only": job.xvec_only}

        sr, wav, nchunks = synth(
            job.text, mode=job.mode, speaker=job.speaker, instruct=job.instruct,
            language=job.language, gap_ms=job.gap_ms, ref=ref,
            on_progress=prog, cancel=lambda: job.cancelled)

        job.events.put(("progress", {"done": nchunks, "total": nchunks, "phase": "encoding"}))
        t0 = time.time()
        audio_bytes, mime = encode_async(sr, wav, job.fmt).result()
        job.audio, job.mime = audio_bytes, mime
        job.meta = {"audio_s": len(wav) / sr, "chunks": nchunks,
                    "wall_s": time.time() - job.started, "encode_s": time.time() - t0,
                    "mode": job.mode, "gpus": N_GPU}
        job.events.put(("done", job.meta))
    except Exception as e:
        job.error = str(e) or type(e).__name__
        job.events.put(("err", {"message": "cancelled" if job.cancelled
                                          else (job.error or "generation failed")}))
    finally:
        if N_GPU:
            try:
                import torch; torch.cuda.empty_cache()
            except Exception: pass
        job.events.put(("__end__", None))

from concurrent.futures import ThreadPoolExecutor
_job_pool = ThreadPoolExecutor(max_workers=2)

# --------------------------------------------------------------------------
# routes
# --------------------------------------------------------------------------
@app.get("/", response_class=HTMLResponse)
def index():
    return HTMLResponse(INDEX_HTML)

@app.get("/manifest.webmanifest")
def manifest():
    return Response(MANIFEST_JSON, media_type="application/manifest+json")

@app.get("/icon.svg")
def icon():
    return Response(ICON_SVG, media_type="image/svg+xml")

@app.get("/sw.js")
def sw():
    return Response(SW_JS, media_type="application/javascript")

@app.get("/api/status")
def status():
    if N_GPU:
        acc = torch.cuda.get_device_name(0).replace("Tesla ", "").replace("NVIDIA ", "")
    else:
        acc = "CPU"
    return {"gpus": N_GPU, "accelerator": acc, "model_size": MODEL_SIZE,
            "modes": list(REPLICAS.keys()), "voices": SPEAKERS, "languages": LANGUAGES,
            "batch": BATCH_PER_GPU}

@app.post("/api/ref")
async def upload_ref(file: UploadFile = File(...)):
    data = await file.read()
    if not data:
        raise HTTPException(400, "empty file")
    if len(data) > 25 * 2**20:
        raise HTTPException(413, "reference clip too large (25 MB cap)")
    # NOTE: no extension check — phone browsers often send nameless blobs;
    # ffmpeg probes the actual content during conversion below.
    try:
        return _save_ref(data, file.filename)
    except Exception as e:
        raise HTTPException(400, f"could not decode audio: {e}")

@app.get("/api/ref/{ref_id}/audio")
def ref_audio(ref_id: str):
    with REFS_LOCK:
        r = REFS.get(ref_id)
    if r is None:
        raise HTTPException(404, "reference not found")
    return Response(open(r["path"], "rb").read(), media_type="audio/wav",
                    headers={"Cache-Control": "no-store"})

@app.post("/api/jobs")
async def submit(req: Request):
    body = await req.json()
    text = (body.get("text") or "").strip()
    if not text:
        raise HTTPException(400, "text is required")
    if len(text) > 500_000:
        raise HTTPException(413, "text too large (500k char cap)")
    mode = body.get("mode") or "custom_voice"
    if mode not in REPLICAS:
        raise HTTPException(400, f"model for mode '{mode}' is not loaded")
    speaker = body.get("speaker") or (SPEAKERS[0]["id"] if SPEAKERS else "")
    if mode == "custom_voice" and not any(s["id"].lower() == speaker.lower() for s in SPEAKERS):
        raise HTTPException(400, f"unknown speaker: {speaker}")
    language = body.get("language") or "Auto"
    if language not in LANGUAGES:
        raise HTTPException(400, f"unsupported language '{language}' (pick one from the dropdown)")
    instruct = (body.get("instruct") or "").strip()

    if mode == "voice_design" and not instruct:
        raise HTTPException(400, "voice design needs a description of the voice")
    if mode == "voice_clone":
        if not body.get("ref_id"):
            raise HTTPException(400, "upload a reference clip first (Clone tab)")
        if not (body.get("ref_text") or "").strip() and not body.get("xvec_only"):
            raise HTTPException(400, "type what the clip says — or tick 'Skip transcript' "
                                     "for a timbre-only clone")
    gap = max(0, min(1500, int(body.get("gap_ms", 160))))
    fmt = (body.get("format") or "wav").lower()
    if fmt not in ("wav", "mp3", "flac", "opus"):
        fmt = "wav"
    job = Job(id=uuid.uuid4().hex, mode=mode, text=text, speaker=speaker, instruct=instruct,
              language=language, gap_ms=gap, fmt=fmt,
              ref_id=body.get("ref_id"), ref_text=(body.get("ref_text") or "").strip(),
              xvec_only=bool(body.get("xvec_only", False)))
    with JOBS_LOCK:
        JOBS[job.id] = job
    _job_pool.submit(_run_job, job)
    return {"job_id": job.id}

@app.get("/api/jobs/{job_id}/events")
async def events(job_id: str, request: Request):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(404, "job not found")
    async def gen():
        loop = asyncio.get_event_loop()
        yield "retry: 3000\n\n"
        while True:
            if await request.is_disconnected():
                break
            try:
                evt, data = await loop.run_in_executor(None, lambda: job.events.get(timeout=1.0))
            except Exception:
                yield ": keepalive\n\n"
                continue
            if evt == "__end__":
                break
            yield f"event: {evt}\ndata: {json.dumps(data)}\n\n"
    headers = {"Cache-Control": "no-cache", "X-Accel-Buffering": "no", "Connection": "keep-alive"}
    return StreamingResponse(gen(), media_type="text/event-stream", headers=headers)

@app.get("/api/jobs/{job_id}/audio")
def get_audio(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(404, "job not found")
    if job.error:
        raise HTTPException(500, job.error)
    if job.audio is None:
        raise HTTPException(425, "not ready")
    hdr = {"X-Meta": json.dumps(job.meta), "Cache-Control": "no-store",
           "Content-Disposition": f'attachment; filename="qwen3tts.{job.fmt}"'}
    return Response(job.audio, media_type=job.mime, headers=hdr)

@app.delete("/api/jobs/{job_id}")
def cancel(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(404, "job not found")
    job.cancelled = True
    return {"ok": True}

def _reap_jobs():
    while True:
        time.sleep(60)
        with JOBS_LOCK:
            if len(JOBS) > 32:
                old = sorted(JOBS.values(), key=lambda j: j.started)[: len(JOBS) - 32]
                for j in old:
                    JOBS.pop(j.id, None)
threading.Thread(target=_reap_jobs, daemon=True).start()

print("FastAPI app defined")

## 🚀 Step 5 — Launch → get your phone URL

Starts the server, opens a Cloudflare quick tunnel, prints your personal `https://…trycloudflare.com` link. **Tap it on your phone** — that's the whole point 🎉

In [ ]:
# ──────────────────────────────────────────────
# STEP 5 · LAUNCH → GET YOUR PHONE URL
# run, wait ~10 s, tap the printed https://…trycloudflare.com link
# ──────────────────────────────────────────────

import threading, uvicorn, subprocess, re, time, sys, shutil

PORT = 7860
_server = None
def _serve():
    global _server
    cfg = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning", access_log=False)
    _server = uvicorn.Server(cfg)
    _server.run()

threading.Thread(target=_serve, daemon=True).start()
time.sleep(2)
print(f"uvicorn on http://0.0.0.0:{PORT}")

# Start cloudflared quick tunnel; scrape stderr for the public URL
if not shutil.which("cloudflared"):
    cf_proc = None
    print("\n⚠ cloudflared not installed (Step 1 download failed?) — "
          f"the app is reachable inside the kernel at http://localhost:{PORT} only")
else:
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )

    URL_RE = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    public_url = None
    t_start = time.time()
    while time.time() - t_start < 60:
        line = cf_proc.stdout.readline()
        if not line: time.sleep(0.1); continue
        sys.stdout.write(line)
        m = URL_RE.search(line)
        if m: public_url = m.group(0); break

if public_url:
    print("\n" + "="*60)
    print(f"  📱  Open this on your phone:  {public_url}")
    print(f"      (also http://localhost:{PORT} from inside the kernel)")
    print("="*60)
    try:
        from IPython.display import display, HTML
        display(HTML(f'<div style="padding:14px 16px;border-radius:12px;background:#0b0d10;color:#fff;font-family:sans-serif">'
                     f'<div style="font-size:12px;opacity:.7">Open on your phone:</div>'
                     f'<a style="color:#8b7cf7;font-size:18px;font-weight:700;word-break:break-all" href="{public_url}" target="_blank">{public_url}</a>'
                     f'</div>'))
    except Exception:
        pass
else:
    print("cloudflared did not report a URL — check its output above.")

## 📊 Step 6 — Benchmark *(optional)*

Naive (1 GPU, one sentence at a time) vs the full pipeline. Expect roughly **5–10×** on multi-sentence scripts.

In [ ]:
# ──────────────────────────────────────────────
# STEP 6 · BENCHMARK (OPTIONAL)
# naive 1-GPU vs the full 2-GPU pipeline
# ──────────────────────────────────────────────

# Benchmark: naive (1 sentence per call, 1 GPU, no pipelining) vs the full
# pipeline (2 GPUs x batch-8, length-sorted, parallel encode).  Edit the text
# to something the length of YOUR typical script for a meaningful number.
BENCH_TEXT = ("The quick brown fox jumps over the lazy dog. " * 24).strip()

import numpy as np, time

def _serial(text):
    parts, sr = [], None
    for c in split_script(text):
        rep = REPLICAS["custom_voice"][GPUS[0]]
        w, sr = rep.generate_custom_voice(text=c, speaker="Aiden", language="English")
        parts.append(np.asarray(w[0], dtype=np.float32))
    return np.concatenate(parts), sr

if "REPLICAS" in globals() and "custom_voice" in REPLICAS:
    t0 = time.time(); _w1, _sr = _serial(BENCH_TEXT);                 t_ser = time.time() - t0
    t0 = time.time(); _sr2, _w2, _n = synth(BENCH_TEXT, mode="custom_voice",
                                            speaker="Aiden", language="English"); t_par = time.time() - t0
    audio_s = len(_w2) / _sr2
    print(f"serial  (1 GPU, batch=1)          : {t_ser:6.2f}s")
    print(f"pipeline({N_GPU} GPU, batch={BATCH_PER_GPU})      : {t_par:6.2f}s   -> {t_ser/t_par:4.2f}x faster")
    print(f"audio: {audio_s:.1f}s  ·  RTF {t_par/audio_s:.3f}  ({audio_s/t_par:.1f}x realtime, {_n} chunks)")
else:
    print("custom_voice model not loaded (run Step 2 first) — nothing to benchmark.")